# Reproducing Yang et al. (2025) from dandiset 001212

Paper. Yang Y, Ni H, Sun Y, Xing Y, Wang C, Zhang J, Zhu J, Xu K (2025). Individualized target
selection of closed-loop electrical stimulation for the treatment of spontaneous temporal lobe
epilepsy. IEEE Transactions on Neural Systems and Rehabilitation Engineering 33:4008-4016,
[doi:10.1109/TNSRE.2025.3614867](https://doi.org/10.1109/TNSRE.2025.3614867). The claims file
(`claims/yang-2025.yaml`) was written from the version of record (open access) and its supplement
(Figures S1 to S3, Table S1).

Dandiset. [DANDI:001212](https://dandiarchive.org/dandiset/001212/0.241023.0931), version
0.241023.0931 (doi:10.48324/dandi.001212/0.241023.0931). The paper cites no DANDI version (it has
no data availability statement), and the dandiset has a single published version, the most recent,
so it is pinned. The draft has the same 56 assets and size. The files are streamed from the
archive; nothing is downloaded to disk.

What the dandiset holds. One NWB file per rat (R1 to R7) and group (4 baseline, 4 stimulation, in
the order 00 to 07); each file holds one 600 s clip per seizure, 4 channels (DG, CA1, CA3, ANT) at
1 kHz in microvolts. There are no seizure onset or end times, no stimulation times, no interictal
recordings and no recording durations. So the seizure onset and end are marked here from the LFP
by a stated rule (an assumption; the paper gives none), the stimulation trains are found from the
artefact they leave in the recording, and the claims on interictal data, seizure frequency and
seizure duration proportion are blocked.

| Claim | Tested here |
|---|---|
| tables1-network-stable-short-term | tested |
| fig2-r1-target-shifts | tested |
| figs2-target-assignment | tested |
| fig3a-target-reduces-each-rat | tested |
| fig3a-nontarget-can-lengthen | tested |
| fig3a-per-rat-site-differences | tested |
| fig3a-target-vs-nontarget-total | tested |
| fig3a-some-nontarget-shorten | tested |
| fig3b-same-site-target-vs-nontarget | tested |
| seizure-duration-proportion | blocked, data missing (no recording time) |
| fig3c-frequency-no-difference | blocked, data missing (no recording time) |
| fig4b-psd-target-bands | tested |
| fig4b-psd-other-bands-unchanged | tested |
| fig5b-iis-post-seizure | tested |
| fig5c-interictal-iis-no-difference | blocked, data missing (no interictal recordings) |
| fig5d-iis-seizure-correlation | blocked, data missing (no interictal recordings, no recording time) |
| fig6cd-interictal-hfo-no-difference | blocked, data missing (no interictal recordings) |
| fig6e-ictal-ripples-no-difference | tested |
| fig6f-ictal-fast-ripples-reduced | tested |

The helper module `yang2025.py` streams each file and reduces each clip, as it is read, to what
the claims use (stimulation trains, seizure onset and end, the GC matrix at onset, PSD ratios,
interictal spike times, HFO counts); its docstrings say where each parameter comes from. Files are
processed four at a time; each worker holds one clip (about 20 MB) at a time.

In [ ]:
import time
import warnings

import numpy as np
import pandas as pd
from scipy import stats

import yang2025 as y

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 400)

started = time.time()
files, clips = y.load_all(workers=4)
print(f"dandiset {y.DANDISET_ID} version {y.DANDISET_VERSION}: {len(files)} files, {len(clips)} clips, "
      f"loaded and reduced in {(time.time() - started) / 60:.1f} min")

In [ ]:
MAIN = (y.SZ_HIGH, y.SZ_LOW)
# Target groups as Figure S2 labels them (claims file, figs2-target-assignment; used by every later claim)
S2_TARGETS = {("R1", "DG"), ("R2", "DG"), ("R2", "CA1"), ("R3", "CA3"), ("R4", "CA3"), ("R5", "ANT"),
              ("R6", "CA3"), ("R7", "ANT")}
# Figure S2, per rat: the targets of baseline groups 1 to 4 and the site stimulated next (claims file)
S2 = {
    "R1": [({"ANT"}, "CA1"), ({"ANT"}, "CA3"), ({"CA3"}, "ANT"), ({"DG"}, "DG")],
    "R2": [({"DG", "CA1"}, "DG"), ({"CA1"}, "ANT"), ({"CA1"}, "CA1"), ({"ANT"}, "CA3")],
    "R3": [({"CA3"}, "CA3"), ({"CA3"}, "ANT"), ({"ANT"}, "DG"), ({"CA3", "ANT"}, "CA1")],
    "R4": [({"CA3"}, "CA1"), ({"CA3"}, "DG"), ({"CA3"}, "ANT"), ({"CA3"}, "CA3")],
    "R5": [({"CA3", "ANT"}, "ANT"), ({"ANT"}, "DG"), ({"ANT"}, "CA1"), ({"ANT"}, "CA3")],
    "R6": [({"ANT"}, "CA1"), ({"CA1", "CA3"}, "ANT"), ({"CA3"}, "DG"), ({"CA3"}, "CA3")],
    "R7": [({"ANT"}, "DG"), ({"ANT"}, "ANT"), ({"ANT"}, "CA1"), ({"ANT"}, "CA3")],
}
RATS = sorted(S2)

rows = []
for c in clips:
    r = {"rat": c["rat"], "gi": c["index"], "group": c["group"], "kind": c["kind"], "site": c["site"],
         "clip": c["clip"], "small": "small" in c["clip"], "n_trains": len(c["trains"])}
    for v in y.SZ_VARIANTS:
        s = c["seizures"][v]
        r[f"dur_{v[0]:g}_{v[1]:g}"] = (s["end"] - s["onset"]) if s else np.nan
    s = c["seizures"][MAIN]
    r["onset"] = s["onset"] if s else np.nan
    r["end"] = s["end"] if s else np.nan
    r["by_train"] = bool(s and s["by_train"])
    r["latency"] = (c["trains"][0][0] - s["onset"]) if (s and c["trains"]) else np.nan
    r["first_train"] = c["trains"][0][0] if c["trains"] else np.nan
    r["last_train_end"] = c["trains"][-1][1] if c["trains"] else np.nan
    # trains that start after the detected end: the device was still detecting seizure activity
    r["trains_after_end"] = sum(t0 > s["end"] for t0, t1 in c["trains"]) if s else 0
    r["trains_in_15_45"] = sum(t1 > s["end"] + 15 and t0 < s["end"] + 45 for t0, t1 in c["trains"]) if s else 0
    r["stimulated"] = bool(s and any(t1 > s["onset"] - 1 and t0 < s["end"] + 1 for t0, t1 in c["trains"]))
    rows.append(r)
sz = pd.DataFrame(rows)
sz["dur"] = sz[f"dur_{MAIN[0]:g}_{MAIN[1]:g}"]
# variant "device end": in stimulation clips the seizure ends no earlier than the last train, as the
# device stimulates while it detects seizure activity (Methods C)
sz["end_dev"] = np.where(sz.kind == "stim", np.fmax(sz.end, sz.last_train_end), sz.end)
sz["dur_dev"] = sz.end_dev - sz.onset
sz["target"] = [(r, s) in S2_TARGETS if k == "stim" else False for r, s, k in zip(sz.rat, sz.site, sz.kind)]
sz["condition"] = np.where(sz.kind == "baseline", "baseline", np.where(sz.target, "target", "non-target"))
DUR_COLS = [f"dur_{a:g}_{b:g}" for a, b in y.SZ_VARIANTS]
print("seizure clips:", len(sz), "; seizure found (main thresholds):", int(sz.dur.notna().sum()))

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
meta = pd.DataFrame([{"rat": f["rat"], "index": f["index"], "group": f["group"], "session_id": f["session_id"],
                      "start": f["session_start_time"][:10], "clips": len(f["clips"]),
                      "first_clip_date": min(y.clip_date(c["name"])[0] for c in f["clips"]),
                      "last_clip_date": max(y.clip_date(c["name"])[0] for c in f["clips"]),
                      "MB": round(f["size"] / 1e6)} for f in files]).sort_values(["rat", "index"])
print("Files per rat:", meta.groupby("rat").size().to_dict())
print("Groups per rat (baseline, stim):", {r: (int((g.group.str.startswith("base")).sum()), int((~g.group.str.startswith("base")).sum()))
                                         for r, g in meta.groupby("rat")})
print(meta.to_string(index=False))
f0 = files[0]
print("\nOne file's contents besides the clips (all files checked below):")
for k in ["top_level", "intervals", "processing", "analysis", "stimulus", "electrode_labels", "electrode_locations",
          "electrode_group_names", "subject", "session_description"]:
    print(f"  {k}: {f0[k]}")
same = {k: len({str(f[k]) for f in files}) for k in ["intervals", "processing", "analysis", "stimulus",
                                                       "electrode_labels", "electrode_locations", "session_description"]}
print("Distinct values over the 56 files:", same)
print("Subjects:", sorted({(f["subject"]["subject_id"], f["subject"]["sex"], f["subject"]["age"], f["subject"]["weight"]) for f in files}))
cl = pd.DataFrame([dict(c, rat=f["rat"], group=f["group"]) for f in files for c in f["clips"]])
print("\nClip shapes:", cl["shape"].astype(str).value_counts().to_dict(), "; dtypes:", cl.dtype.value_counts().to_dict(),
      "; units:", cl.unit.value_counts().to_dict(), "; conversion:", cl.conversion.unique().tolist(),
      "; rate (Hz):", cl.rate.unique().tolist(), "; starting_time:", cl.starting_time.unique().tolist())
print("Clips per group: min", int(meta.clips.min()), "median", float(meta.clips.median()), "max", int(meta.clips.max()),
      "; groups with fewer than 6 clips (Methods C: no less than 6 seizures per group):",
      meta[meta.clips < 6][["rat", "group", "clips"]].values.tolist())
print("Clip name suffixes:", cl.name.str.extract(r"ictal\d+(-\w+)?$")[0].fillna("(none)").value_counts().to_dict())

The dandiset holds 7 rats with 4 baseline and 4 stimulation groups each, as its description and
Methods C say, and the stimulated site is in each stimulation group's file name. Every file holds
only the clips and the electrodes table: no intervals, processing, analysis or stimulus data, so no
seizure times, stimulation times or recording durations. The electrodes table is the same in all
files (DG, CA1, CA3, ANT, in that order); the coordinates match Methods A except the ANT depth
(DV -5.6 mm here, -6.0 mm in the paper). All subjects are given as P2M and 300-350 g in every
session, though the sessions span up to four months per rat (Methods A: 7-9 weeks and 250-300 g at
the start). Two groups hold fewer than the 6 seizures Methods C states (R1 base2, 5; R7 CA3-stim,
4). Twenty clips are named `-small` and two `-large`, which nothing in the dandiset explains; they
are kept, with a variant without them where it matters. The dates in the clip names do not always
follow the group order: R3's CA1-stim group (07) holds clips dated 18 to 20 October, within the
dates of its DG-stim group (05) and before its base3 group (06), and R5's base0 runs from 8
September to 25 October.

In [ ]:
vals = pd.DataFrame([{"rat": c["rat"], "group": c["group"], "clip": c["clip"], "nan": c["n_nan"],
                      **{f"min_{ch}": c["min"][i] for i, ch in enumerate(y.CHANNELS)},
                      **{f"max_{ch}": c["max"][i] for i, ch in enumerate(y.CHANNELS)},
                      **{f"sd_{ch}": c["sd"][i] for i, ch in enumerate(y.CHANNELS)}} for c in clips])
print("NaN samples:", int(vals.nan.sum()))
print("Range over all clips (microvolts):")
print(vals[[c for c in vals if c[:3] in ("min", "max")]].agg(["min", "median", "max"]).round(0).to_string())
print("SD per channel (microvolts), median over each rat's clips:")
print(vals.groupby("rat")[[f"sd_{ch}" for ch in y.CHANNELS]].median().round(1).to_string())
md5 = pd.Series([c["md5"] for c in clips])
print("\nIdentical clips (same MD5 of all samples):", int(md5.duplicated().sum()))
# clips whose names (date and seizure number) occur in two groups of a rat
cl["key"] = cl.rat + ":" + cl.name.str.replace("-small", "").str.replace("-large", "")
dups = cl[cl.key.duplicated(keep=False)].sort_values("key")
print("Clip names found in two groups of the same rat:", len(dups) // 2)
env = {(c["group"], c["rat"], c["clip"]): c["ll_env"] for c in clips}


def env_corr(a, b):
    a = (a - a.mean()) / a.std(); b = (b - b.mean()) / b.std(); n = len(a)
    cc = np.fft.irfft(np.fft.rfft(a, 2 * n) * np.conj(np.fft.rfft(b, 2 * n))) / n
    return cc.max()


pairs = []
for k, g in dups.groupby("key"):
    (_, a), (_, b) = g.iterrows()
    pairs.append({"rat": a.rat, "name": a["name"], "group_a": a.group, "group_b": b.group,
                  "envelope_r": round(env_corr(env[(a.group, a.rat, a["name"])], env[(b.group, b.rat, b["name"])]), 3)})
print(pd.DataFrame(pairs).to_string(index=False))
allr = []
keys = list(env)
for i in range(len(keys)):
    for j in range(i + 1, len(keys)):
        if keys[i][1] == keys[j][1]:
            allr.append(env_corr(env[keys[i]], env[keys[j]]))
print(f"Largest envelope correlation over all {len(allr)} pairs of clips of the same rat: {max(allr):.3f}; "
      f"99th percentile {np.percentile(allr, 99):.3f}")

The values are finite, in microvolts, and clipped near +-6390 uV (the amplifier's range) in some
clips. No two clips are identical. Sixteen clip names (date and seizure number) occur in two groups
of the same rat, but their contents differ: the correlation of their line-length envelopes (0.24
to 0.85, at the best lag) is below the 99th percentile over all pairs of clips of the same rat, so
these are different seizures under one name, not one recording deposited twice. Each clip is
counted once. Units are not followed across sessions; the channels are the same electrodes in all
files of a rat by the electrodes table, which is an assumption the analysis shares with the paper.

In [ ]:
tr = pd.DataFrame([{"rat": c["rat"], "group": c["group"], "kind": c["kind"], "clip": c["clip"], "start": t0, "end": t1}
                   for c in clips for t0, t1 in c["trains"]])
tr["length"] = tr.end - tr.start
tr["gap_to_next"] = tr.groupby(["rat", "group", "clip"]).start.shift(-1) - tr.end
print("Detected stimulation trains:", len(tr), "; in baseline groups:", int((tr.kind == "baseline").sum()))
print("Train length (s) by group kind:")
print(tr.groupby("kind").length.describe().round(2).to_string())
print("Pause between trains (s):", tr.gap_to_next.describe().round(2).to_dict())
print("\nClips with at least one train, by group kind:", sz.groupby("kind").apply(lambda g: f"{(g.n_trains > 0).sum()} of {len(g)}").to_dict())
print("Stimulation-group clips without a train:", sz[(sz.kind == "stim") & (sz.n_trains == 0)][["rat", "group", "clip"]].values.tolist())
print("Baseline-group clips with trains:")
print(sz[(sz.kind == "baseline") & (sz.n_trains > 0)][["rat", "group", "clip", "n_trains", "onset", "latency"]].round(1).to_string(index=False))
print(tr[tr.kind == "baseline"].round(2).to_string(index=False))

Stimulation trains are found from their artefact (`yang2025.find_trains`). In the stimulation
groups, 250 of 252 clips carry trains; they last 5 s (median) and the pause between trains is
5.5 s (median), as Methods C states (5 s trains, 5 s pause). The two stimulation clips without a
train are in R7's CA3 group, where the artefact is weakest. This confirms that the clips of the
stimulation groups are stimulated seizures and gives each one its stimulation times.

Eleven baseline clips also carry trains. In R1 base2 (8/29 ictal1, 8/30 ictal1), R4 base0 (9/6
ictal1 to 3), R5 base0 (10/25 ictal1) and R6 base1 (9/17 ictal1), they are 4.25 to 6 s trains,
3.6 to 4.9 s after the detected onset in three of them, like those of the stimulation groups: these baseline
seizures were stimulated. The single trains of 2 s (R3 9/11 ictal7, R6 9/12 ictal9), the 7.5 s one
(R6 9/12 ictal3) and the irregular ones of R7 2024/1/3 ictal1 (2 to 10.5 s, before the seizure) may be
other artefacts. The groups are analysed as deposited, with a variant that leaves out every
baseline clip with a train.

In [ ]:
print("Seizure found per clip, by threshold pair (high, low):",
      {c: int(sz[c].notna().sum()) for c in DUR_COLS}, "of", len(sz))
st = sz[(sz.kind == "stim") & sz.latency.notna()]
print(f"\nStimulation-group seizures with trains: {len(st)}. First train minus detected onset (s):")
print(st.latency.describe().round(2).to_string())
print("Negative (train before onset):", int((st.latency < 0).sum()), "; over 15 s:", int((st.latency > 15).sum()),
      "; seizure chosen as the run that holds the first train:", int(st.by_train.sum()))
print("By rat (median latency, s):", st.groupby("rat").latency.median().round(1).to_dict())
# Contrast: line length envelope around the first train in stimulation clips, and around the same
# clip times in baseline clips. The closed-loop detector starts a train on seizure onset, so the
# envelope must rise before the train in stimulation clips.
rel = np.arange(-20, 21, 5)
cen = (np.arange(0, 600000 - y.LL_WIN, y.LL_STEP) + y.LL_WIN / 2) / y.FS
prof_stim = []
for c in clips:
    if c["kind"] == "stim" and c["trains"]:
        t = c["trains"][0][0]
        prof_stim.append([np.interp(t + d, cen, c["ll_env"]) for d in rel])
print("\nMedian line-length score (max over channels, x median) relative to the first train, stimulation clips:")
print(pd.Series(np.median(prof_stim, 0), index=[f"{d:+d}s" for d in rel]).round(2).to_string())
base_rand = []
rng = np.random.default_rng(0)
for c in clips:
    if c["kind"] == "baseline":
        t = rng.uniform(30, 570)
        base_rand.append([np.interp(t + d, cen, c["ll_env"]) for d in rel])
print("Same profile at a random time in each baseline clip:")
print(pd.Series(np.median(base_rand, 0), index=[f"{d:+d}s" for d in rel]).round(2).to_string())
print("\nSeizure duration (s), main thresholds, by condition:")
print(sz.groupby("condition").dur.describe().round(1).to_string())
print("Seizure onset in the clip (s):", sz.onset.describe().round(0).to_dict())

In [ ]:
# Checks of the onset and end rule asked for in verification round 1.
print("Detected onset in the clip (s), counts per 20 s bin (main thresholds):")
print(pd.cut(sz.onset, [0, 200, 240, 260, 280, 300, 320, 340, 360, 400, 600]).value_counts().sort_index().to_string())
# If the clips had been cut around the authors' own onset at 300 s, the first train, which the
# closed-loop detector starts on seizure onset (Methods C), would follow 300 s closely.
ft = sz[(sz.kind == "stim") & sz.first_train.notna()]
print(f"\nStimulation clips, absolute time of the first train (s): {ft.first_train.describe().round(1).to_dict()}")
print("  within 300 to 305 s:", int(ft.first_train.between(300, 305).sum()), "of", len(ft),
      "; within 295 to 315 s:", int(ft.first_train.between(295, 315).sum()))
print("  by rat (median, IQR):", {r: (round(g.first_train.median(), 1), round(g.first_train.quantile(.25), 1),
                                     round(g.first_train.quantile(.75), 1)) for r, g in ft.groupby("rat")})
# Does the end come from the device? Detected end minus the end of the last train, stimulation clips
se = sz[(sz.kind == "stim") & sz.end.notna() & sz.last_train_end.notna()].copy()
se["end_minus_last_train"] = se.end - se.last_train_end
print("\nDetected end minus end of the last train (s), stimulation clips, by condition:")
print(se.groupby("condition").end_minus_last_train.describe().round(1).to_string())
print("Within 2 s of the last train's end:", se.groupby("condition").apply(
    lambda g: f"{int((g.end_minus_last_train.abs() <= 2).sum())} of {len(g)}").to_dict())
print("\nTrains that start after the detected end, and trains overlapping 15 to 45 s after the end (the IIS window):")
print(se.groupby(["condition"]).agg(seizures=("clip", "size"), with_train_after_end=("trains_after_end", lambda v: int((v > 0).sum())),
                                    with_train_in_15_45=("trains_in_15_45", lambda v: int((v > 0).sum())),
                                    trains_in_15_45=("trains_in_15_45", "sum")).to_string())
print(se.groupby(["rat", "condition"]).trains_in_15_45.apply(lambda v: int((v > 0).sum())).unstack().to_string())
print("Durations with the device end (stimulation clips end no earlier than the last train), by condition:")
print(sz.groupby("condition").dur_dev.describe().round(1).to_string())


In [ ]:
# PSD and HFO windows of the stimulation clips under the three window readings (yang2025.psd_windows):
# how many post windows overlap a train or lie within 0.5 s of one, and how long the pre windows are.
wrow = []
for c in clips:
    if c["kind"] != "stim" or "psd_windows" not in c:
        continue
    cond = "target" if (c["rat"], c["site"]) in S2_TARGETS else "non-target"
    for reading, w in c["psd_windows"].items():
        if w is None:
            continue
        (a0, a1), (b0, b1) = w
        later = [(t0, t1) for t0, t1 in c["trains"] if t0 > a1 + 1]  # the trains after the first
        overlap = any(t0 < b1 and t1 > b0 for t0, t1 in later)
        close = any(t0 < b1 + 0.5 and t1 > b0 - 0.5 for t0, t1 in later)
        wrow.append({"reading": reading, "condition": cond, "pre_len": a1 - a0, "post_len": b1 - b0,
                     "post_overlaps_train": overlap, "post_within_0.5s_of_train": close})
wd = pd.DataFrame(wrow)
print("Post windows against the trains after the first (the first train ends where the post window starts):")
print(wd.groupby(["reading", "condition"]).agg(n=("pre_len", "size"), post_overlaps_train=("post_overlaps_train", "sum"),
      post_within_half_s=("post_within_0.5s_of_train", "sum"), post_len_median=("post_len", "median"),
      post_shorter_than_5s=("post_len", lambda v: int((v < 4.99).sum())), pre_len_median=("pre_len", "median"),
      pre_over_15s=("pre_len", lambda v: int((v > 15).sum())), pre_max=("pre_len", "max")).round(2).to_string())


In [ ]:
# Example clips: LFP, the line-length score and the detected onset, end and trains, for the first clip
# of three groups of R7 (baseline, target, non-target) and of R4 (where the first train comes late).
import matplotlib.pyplot as plt

EXAMPLES = [("R7", "base0"), ("R7", "ANT-stim"), ("R7", "DG-stim"), ("R4", "CA3-stim")]
fig, axes = plt.subplots(len(EXAMPLES), 1, figsize=(11, 2.6 * len(EXAMPLES)))
for ax, (rat, group) in zip(axes, EXAMPLES):
    f = next(f for f in files if f["rat"] == rat and f["group"] == group)
    name = f["clips"][0]["name"]
    c = next(c for c in clips if c["rat"] == rat and c["group"] == group and c["clip"] == name)
    s = c["seizures"][MAIN]
    h = y.open_asset(f["asset_id"])
    x = h["acquisition"][name]["data"][:]
    h.close()
    t0, t1 = s["onset"] - 60, s["end"] + 60
    i0, i1 = int(t0 * y.FS), int(t1 * y.FS)
    tt = np.arange(i0, i1) / y.FS
    off = 0
    for ch in range(4):
        seg = x[i0:i1, ch] - np.median(x[i0:i1, ch])
        ax.plot(tt, seg / 1000 + off, lw=0.3, color="0.3")
        off -= 2
    cen = (np.arange(0, len(x) - y.LL_WIN, y.LL_STEP) + y.LL_WIN / 2) / y.FS
    m = (cen >= t0) & (cen <= t1)
    ax2 = ax.twinx()
    ax2.plot(cen[m], c["ll_env"][m], color="tab:blue", lw=1)
    ax2.axhline(y.SZ_HIGH, color="tab:blue", ls=":", lw=0.8); ax2.axhline(y.SZ_LOW, color="tab:blue", ls=":", lw=0.5)
    ax2.set_ylabel("line-length score", color="tab:blue")
    for a, b in c["trains"]:
        if b > t0 and a < t1:
            ax.axvspan(a, b, color="tab:orange", alpha=0.3, lw=0)
    ax.axvline(s["onset"], color="tab:green"); ax.axvline(s["end"], color="tab:red")
    ax.set_xlim(t0, t1); ax.set_yticks([]); ax.set_title(f"{rat} {group} {name}: onset {s['onset']:.1f} s, end {s['end']:.1f} s", fontsize=9)
    print(f"{rat} {group} {name}: LFP range {x[i0:i1].min():.0f} to {x[i0:i1].max():.0f} uV (traces in mV, offset by 2 mV); "
          f"score range {c['ll_env'][m].min():.2f} to {c['ll_env'][m].max():.2f}; trains in view "
          f"{[(round(a, 1), round(b, 1)) for a, b in c['trains'] if b > t0 and a < t1]}")
    del x
axes[-1].set_xlabel("time in clip (s)")
fig.suptitle("Green: detected onset; red: detected end; orange: detected trains; blue: line-length score (dotted: thresholds)", fontsize=9)
fig.tight_layout()
plt.show()


The onset and end rule (assumption, `yang2025.detect_seizure`) finds a seizure in 517 of 519 clips
at the main thresholds. In the stimulation groups the first train follows the detected onset by a
median 6.5 s (25th to 75th percentile 2.9 to 15.6 s); in 10 of 248 the train comes first, and the
median is 2.9 to 7.1 s in R1, R2, R3, R6 and R7 and 15 to 17 s in R4 and R5. The line-length score is
1.8 times its median 5 s before the first train and 1.3 to 1.5 times 10 to 20 s before it, while at
random times in baseline clips it stays at 1.1 to 1.2; the 0 and +5 s points include the train
artefact. The closed-loop detector starts stimulation on seizure onset (Methods C), so this
contrast shows that the clips' clock, the trains and the seizures agree, and that the detected
onsets fall a few seconds before the detector's, more so in R4 and R5. Detected durations are 6 to
207 s, median 41 to 45 s in each condition.

Checks added after verification round 1 (the three cells above).

- **Were the clips cut around the authors' own onset?** The detected onsets cluster near the clip
  centre (median 301 s; 273 of 517 between 280 and 320 s), so the clips might have been cut at an
  onset the authors marked at 300 s, which would in effect be deposited onset times (claims file,
  reading a). If so, the first train, which the closed-loop detector starts on seizure onset
  (Methods C), would follow 300 s by a few seconds. It does not: in the 250 stimulation clips with a
  train, the first train falls at a median 309 s (25th to 75th percentile 292 to 328 s), and only 21
  fall between 300 and 305 s (69 between 295 and 315 s). The clips are centred on the seizure only
  roughly, so 300 s cannot stand in for an onset time, and that reading is not run.
- **Is the end set by the device?** In stimulation clips the detected end comes a median 10.1 s
  (non-target) and 7.6 s (target) after the last train ends, and within 2 s of it in only 18 of 163
  and 6 of 85 seizures. A train starts after the detected end in 28 non-target and 16 target
  seizures, and overlaps the 15 to 45 s IIS window in 6 and 1 (4 of the 6 in R7). Each duration claim and the
  IIS claim are also shown with a "device end" variant, in which a stimulation seizure ends no earlier
  than its last train.
- **Do the PSD and HFO windows hold train artefact?** As Methods D is worded (reading "paper"), 8
  non-target and 1 target post windows overlap the next train. The main reading is now "guarded"
  (`yang2025.psd_windows`): each window keeps 0.25 s (one step of the train detector) clear of the
  trains and the post window is cut where the next train starts. No guarded post window overlaps a
  train; 41 non-target and 35 target ones are shorter than 5 s. The pre window, from the detected
  onset to the first train, is longer than 15 s in 48 non-target and 20 target seizures (up to 314 s),
  so a third reading ("guarded_pre5") limits it to the 5 s before the first train.
- **The rule on example clips.** The figure shows the LFP, the line-length score, the detected onset
  and end and the trains in four clips (R7 baseline, target and non-target, and R4, where the first
  train comes late). The rule remains an assumption; nothing in the dandiset can validate it beyond
  the train-latency contrast above.

## Granger causality at seizure onset

For each baseline seizure, conditional GC between the 4 channels over the first 5 s after onset,
in 9 windows of 1 s with 0.5 s overlap (Methods B), from VAR models fitted by OLS (the MVGC
toolbox's time-domain estimator, reimplemented in `yang2025.conditional_gc`). The model order is
chosen per window by BIC, with AIC as a variant, up to 20; as AIC reaches that cap in 422 of 2403
windows, AIC is also run with the cap raised to 30 (`gc_aic30`; 53 windows reach 30). cf_w is computed with both signs; as it
is linear in the GC matrix, the two window-combination readings of the claims file give the same
value.

In [ ]:
# cf_w vectors of the baseline seizures (Methods B and C), per reading of the sign and the order criterion
base_clips = [c for c in clips if c["kind"] == "baseline" and "gc_bic" in c]
print("Baseline seizures with a GC matrix:", len(base_clips), "of", int((sz.kind == "baseline").sum()),
      "; dropped (no seizure found, or fewer than 5 s after onset in the clip):",
      int((sz.kind == "baseline").sum()) - len(base_clips))
orders = pd.Series([p for c in base_clips for p in c["orders_bic"]])
print("VAR model order by BIC (per window):", orders.describe().round(1).to_dict(),
      "; at the cap of", y.GC_MAX_ORDER, ":", int((orders == y.GC_MAX_ORDER).sum()))
orders_a = pd.Series([p for c in base_clips for p in c["orders_aic"]])
print("VAR model order by AIC (per window):", orders_a.describe().round(1).to_dict(),
      "; at the cap:", int((orders_a == y.GC_MAX_ORDER).sum()))
orders_a30 = pd.Series([p for c in base_clips for p in c["orders_aic30"]])
print("VAR model order by AIC with the cap raised to 30 (per window):", orders_a30.describe().round(1).to_dict(),
      "; at 30:", int((orders_a30 == 30).sum()), "; above 20:", int((orders_a30 > 20).sum()))
gcm = np.array([c["gc_bic"] for c in base_clips])
print("GC values (BIC order), off-diagonal, range:", float(gcm[:, ~np.eye(4, dtype=bool)].min().round(4)), "to",
      float(gcm[:, ~np.eye(4, dtype=bool)].max().round(4)), "; median", float(np.median(gcm[:, ~np.eye(4, dtype=bool)]).round(4)))
SIGNS = ["out_minus_in", "in_minus_out"]
ORDERS = ["gc_bic", "gc_aic", "gc_aic30"]
cf = {(s, o): {} for s in SIGNS for o in ORDERS}
for c in base_clips:
    for s in SIGNS:
        for o in ORDERS:
            cf[(s, o)].setdefault((c["rat"], c["index"]), []).append(y.cfw(c[o], s))
cf = {k: {g: np.array(v) for g, v in d.items()} for k, d in cf.items()}
small_flag = {(c["rat"], c["index"]): [] for c in base_clips}
for c in base_clips:
    small_flag[(c["rat"], c["index"])].append("small" in c["clip"])

### tables1-network-stable-short-term

> In every rat the cf_w vectors of seizure onset correlate more strongly between seizures of the
> same baseline group than between seizures of different baseline groups (Pearson r, intra-group
> against inter-group, Mann-Whitney).

Criterion: under one reading, intra-group r above inter-group r with Mann-Whitney p < 0.05 in all
7 rats, and at least 10 of the 14 means within the Table S1 SDs. Readings: (a) each pair once or
(b) twice; self-pairs (c) excluded or (d) included; BIC or AIC order.

In [ ]:
TS1 = {"R1": (0.72, 0.30, -0.11, 0.55), "R2": (0.47, 0.57, 0.09, 0.56), "R3": (0.77, 0.19, 0.59, 0.34),
       "R4": (0.81, 0.46, 0.79, 0.46), "R5": (0.74, 0.34, 0.71, 0.28), "R6": (0.61, 0.34, 0.36, 0.47),
       "R7": (0.81, 0.36, 0.71, 0.41)}  # Table S1, as the claims file gives it: intra mean, SD, inter mean, SD


def r_pairs(groups, pairing, self_pairs):
    """Intra- and inter-group Pearson r between the cf_w vectors of a rat's baseline seizures."""
    vecs = np.vstack([groups[g] for g in sorted(groups)])
    lab = np.concatenate([[g] * len(groups[g]) for g in sorted(groups)])
    R = np.corrcoef(vecs)
    n = len(vecs)
    intra, inter = [], []
    for i in range(n):
        for j in range(n):
            if i == j:
                if self_pairs:
                    intra.append(1.0)
                continue
            if pairing == "once" and j < i:
                continue
            (intra if lab[i] == lab[j] else inter).append(R[i, j])
    return np.array(intra), np.array(inter)


res = []
for o in ORDERS:
    for pairing in ["once", "twice"]:
        for self_pairs in [False, True]:
            for rat in RATS:
                groups = {g[1]: v for g, v in cf[("out_minus_in", o)].items() if g[0] == rat}
                a, b = r_pairs(groups, pairing, self_pairs)
                p = stats.mannwhitneyu(a, b, alternative="two-sided").pvalue
                ti, si, te, se = TS1[rat]
                res.append({"order": o, "pairs": pairing, "self": self_pairs, "rat": rat, "n_intra": len(a), "n_inter": len(b),
                            "intra_r": a.mean(), "intra_sd": a.std(ddof=1), "inter_r": b.mean(), "inter_sd": b.std(ddof=1),
                            "p_MW": p, "intra_ok": abs(a.mean() - ti) <= si, "inter_ok": abs(b.mean() - te) <= se,
                            "paper_intra": ti, "paper_inter": te})
t1 = pd.DataFrame(res)
t1["holds"] = (t1.intra_r > t1.inter_r) & (t1.p_MW < 0.05)
print("The cf_w sign does not change Pearson r (r(-a,-b) = r(a,b)), so one sign is shown.")
print("Reading: BIC order, each pair once, self-pairs excluded")
print(t1[(t1.order == "gc_bic") & (t1.pairs == "once") & (~t1.self)].round(3).to_string(index=False))
print("\nReading: BIC order, each pair once, self-pairs (r = 1) included in the intra-group set")
print(t1[(t1.order == "gc_bic") & (t1.pairs == "once") & (t1.self)].round(3).to_string(index=False))
summ = t1.groupby(["order", "pairs", "self"]).agg(rats_holding=("holds", "sum"), means_within_tol=("intra_ok", "sum"))
summ["means_within_tol"] += t1.groupby(["order", "pairs", "self"]).inter_ok.sum()
print("\nPer reading: rats where intra > inter with MW p < 0.05 (of 7), and means within the Table S1 SD (of 14):")
print(summ.to_string())
# variant: without clips named -small
res2 = []
for rat in RATS:
    groups = {g[1]: v[~np.array(small_flag[g])] for g, v in cf[("out_minus_in", "gc_bic")].items() if g[0] == rat}
    a, b = r_pairs(groups, "once", False)
    res2.append({"rat": rat, "intra_r": a.mean(), "inter_r": b.mean(), "p_MW": stats.mannwhitneyu(a, b).pvalue})
print("\nVariant without '-small' clips (BIC, pairs once, no self-pairs):")
print(pd.DataFrame(res2).round(4).to_string(index=False))

**Result: reproduced under reading (d).** With self-pairs included (each pair once, BIC order),
intra-group r exceeds inter-group r with p < 0.05 in all 7 rats, and 11 of the 14 means lie within
the Table S1 SDs; the same holds with AIC order capped at 20 or 30. With self-pairs excluded, R4
(0.154 against 0.208, p = 0.24) and R7 (0.534 against 0.514, p = 0.15) do not hold, so 5 of 7 rats
hold and 10 of 14 means are within tolerance, under every order criterion (6 of 7 with each pair
counted twice). The outcome therefore rests on counting the self-pairs (r = 1) as intra-group
pairs, a reading the claims file names; they add values of 1 to the intra-group set and favour the
claim. With self-pairs excluded, the intra-group means are lower than Table S1's in every rat but
R6 (for example R1 0.445 against 0.72). Leaving out the `-small` clips changes no rat's result. The
outcome also rests on the onset rule, which is an assumption.

### fig2-r1-target-shifts

> In rat R1 the node with the highest cf_w at seizure onset changes across the four baseline
> groups (ANT, ANT, CA3, DG), while the cf_w pattern is consistent within each group.

Criterion: under one reading, the node with the highest mean cf_w in each of R1's baseline groups is
the one named; partially reproduced when three of four match.

In [ ]:
print("R1: mean cf_w per node in each baseline group, and the node with the highest mean")
fig2 = []
for s in SIGNS:
    for o in ORDERS:
        for gi, want in zip([0, 2, 4, 6], ["ANT", "ANT", "CA3", "DG"]):
            v = cf[(s, o)][("R1", gi)]
            fig2.append({"sign": s, "order": o, "group": f"base{gi // 2}", "n": len(v),
                         **{ch: v.mean(0)[i] for i, ch in enumerate(y.CHANNELS)},
                         "top": y.CHANNELS[int(np.argmax(v.mean(0)))], "paper": want})
fig2 = pd.DataFrame(fig2)
fig2["match"] = fig2.top == fig2.paper
print(fig2.round(4).to_string(index=False))
print("\nMatches of 4, per reading:", fig2.groupby(["sign", "order"]).match.sum().to_dict())
# consistency within groups: mean Pearson r of R1 cf_w vectors within each group
for gi in [0, 2, 4, 6]:
    v = cf[("out_minus_in", "gc_bic")][("R1", gi)]
    R = np.corrcoef(v)
    print(f"R1 base{gi // 2}: {len(v)} seizures, mean within-group r {R[np.triu_indices(len(v), 1)].mean():.2f}")
print("Figure 2A shows 9, 9, 7 and 9 seizures; here", [len(cf[("out_minus_in", "gc_bic")][("R1", g)]) for g in [0, 2, 4, 6]])

**Result: partially reproduced.** With cf_w as out-strength minus in-strength and AIC order, the
top nodes are CA1, ANT, CA3, DG: 3 of the 4 named (base0 gives CA1 where the paper has ANT), with
the cap at 20 or at 30. With BIC order 2 of 4 match, and with the other sign none. The top node
does change across groups, and the mean within-group r of the cf_w vectors is 0.26 to 0.66. R1's
groups hold 9, 10, 5 and 11 seizures, where Figure 2A shows 9, 9, 7 and 9.

### figs2-target-assignment

> Across the 7 rats, the targets computed from each baseline group's seizures, compared with the
> site stimulated in the next group, give 8 target and 20 non-target stimulation groups as Figure
> S2 shows.

Criterion: reproduced when, under one reading, at least 24 of the 28 baseline groups give the
Figure S2 target set and all 28 target and non-target labels agree; partially reproduced when at
least 18 baseline groups match. Readings: cf_w sign, order criterion, and (a) the node with the
largest mean or (b) the nodes significantly high (Kruskal-Wallis, then Mann-Whitney against the top
node). The site stimulated next and the group order are the dataset's.

In [ ]:
def computed_targets(s, o, reading):
    out = {}
    for (rat, gi), v in cf[(s, o)].items():
        out[(rat, gi)] = y.targets_top(v) if reading == "a" else y.targets_test(v)
    return out


stim_groups = sz[sz.kind == "stim"].groupby(["rat", "gi"]).site.first()
rows = []
for s in SIGNS:
    for o in ORDERS:
        for reading in ["a", "b"]:
            T = computed_targets(s, o, reading)
            nb = 0
            ntar = 0
            lab_ok = 0
            for rat in RATS:
                for k in range(4):
                    paper_set, paper_site = S2[rat][k]
                    got = T.get((rat, 2 * k), set())
                    ok = (got == paper_set) or (reading == "a" and len(got & paper_set) == 1 and len(got) == 1)
                    nb += ok
                    site = stim_groups[(rat, 2 * k + 1)]
                    is_t = site in got
                    ntar += is_t
                    lab_ok += is_t == ((rat, site) in S2_TARGETS)
            rows.append({"sign": s, "order": o, "reading": reading, "baseline_groups_matching": nb,
                         "target_groups": ntar, "non_target_groups": 28 - ntar, "labels_agreeing": lab_ok})
s2 = pd.DataFrame(rows)
print("Of 28 baseline groups, those whose computed target set matches Figure S2; target and non-target counts of the")
print("28 stimulation groups from the computed targets, and how many labels agree with Figure S2:")
print(s2.to_string(index=False))
best = s2.sort_values(["baseline_groups_matching", "labels_agreeing"], ascending=False).iloc[0]
print("\nBest reading:", best.to_dict())
T = computed_targets(best.sign, best.order, best.reading)
tab = []
for rat in RATS:
    for k in range(4):
        tab.append({"rat": rat, "baseline": f"base{k}", "computed": ",".join(sorted(T.get((rat, 2 * k), set()))) or "-",
                    "figure_S2": ",".join(sorted(S2[rat][k][0])), "next_site_dataset": stim_groups[(rat, 2 * k + 1)],
                    "next_site_S2": S2[rat][k][1]})
tab = pd.DataFrame(tab)
print(tab.to_string(index=False))
for o in ORDERS:
    Tb = computed_targets("out_minus_in", o, "a")
    print(f"\nComputed targets, reading (a), out minus in, {o}, per rat (base0 to base3) against Figure S2:")
    for rat in RATS:
        print(f"  {rat}:", [(",".join(sorted(Tb[(rat, 2 * k)])), ",".join(sorted(S2[rat][k][0]))) for k in range(4)])
print("\nGroups where the dataset's stimulation order differs from Figure S2:",
      tab[tab.next_site_dataset != tab.next_site_S2][["rat", "baseline", "next_site_dataset", "next_site_S2"]].values.tolist())

**Result: partially reproduced.** Under reading (a), cf_w as out minus in and AIC order with the
cap at 30, 21 of the 28 baseline groups give the Figure S2 target, and the computed labels give 8
target and 20 non-target stimulation groups, as Figure S2 does, though only 22 of the 28 labels
agree with it. With the cap at 20, 20 groups match (9 target, 19 non-target, 21 labels agreeing),
and with BIC order 16, below the 18 the partial outcome needs; the outcome therefore rests on AIC
order. Reading (b) matches 6 to 9 groups, and the other sign 0 or 1, so the data support cf_w as
causal outflow (out minus in), as Figure 2 and the Discussion describe it, against the wording of
Methods B. The misses under AIC with the cap at 30 are R1 base0, R2 base0, R5 base1 to base3 (CA3
computed where Figure S2 has ANT) and R6 base1 and base3. The dataset's stimulation order
differs from Figure S2 in R5 (dataset CA1, ANT, DG, CA3; Figure S2 ANT, DG, CA1, CA3) and R6
(dataset ANT, DG, CA1, CA3; Figure S2 CA1, ANT, DG, CA3), so in these rats the baseline group
before a site is not the one Figure S2 pairs it with. Later claims use the Figure S2 labels, as the
criterion says.

## Seizure duration

Durations are the detected end minus onset (assumption). Each stimulation group's seizures are
normalized to the baseline group just before it in the dataset's order (Results B), under the three
readings of the claims file: relative change (duration over the baseline mean, minus 1),
difference from the baseline mean (s), and z-score against the baseline seizures. Target groups are
those of Figure S2 (R1 DG, R2 DG and CA1, R3 CA3, R4 CA3, R5 ANT, R6 CA3, R7 ANT). Every claim is
shown also for the seizures that received a train (stimulated only), without the baseline clips that
carry trains, at the other two threshold pairs of the onset rule, with each site normalized to the
baseline group Figure S2 pairs it with (which differs from the dataset's order in R5 and R6), and
with the device end (a stimulation seizure ends no earlier than its last train).

In [ ]:
def dunn(values, groups, adjust):
    """Dunn's test after Kruskal-Wallis (tie-corrected), all pairs; Bonferroni when adjust."""
    v = np.asarray(values); g = np.asarray(groups)
    ranks = stats.rankdata(v)
    N = len(v)
    _, t = np.unique(v, return_counts=True)
    tie = 1 - (t ** 3 - t).sum() / (N ** 3 - N)
    labs = sorted(set(g))
    out = {}
    m = len(labs) * (len(labs) - 1) / 2
    for i in range(len(labs)):
        for j in range(i + 1, len(labs)):
            a, b = g == labs[i], g == labs[j]
            se = np.sqrt(tie * N * (N + 1) / 12 * (1 / a.sum() + 1 / b.sum()))
            z = (ranks[a].mean() - ranks[b].mean()) / se
            p = 2 * stats.norm.sf(abs(z))
            out[(labs[i], labs[j])] = min(1.0, p * m) if adjust else p
    return out


NORMS = ["relative", "difference", "zscore"]


# Figure S2 pairs each stimulation site with the baseline group before it; in R5 and R6 the dataset's
# order differs, so a variant takes the baseline group that Figure S2 pairs with each site.
S2_BASE = {(rat, site): 2 * k for rat in RATS for k, (_, site) in enumerate(S2[rat])}


def normalized(df, col="dur", stim_only=False, drop_base_trains=False, s2_order=False):
    """Per-seizure normalized durations of the stimulation groups, each against the baseline group
    just before it (Results B: "Each baseline group served as a reference for subsequent
    stimulation groups")."""
    out = []
    for (rat, gi), g in df[df.kind == "stim"].groupby(["rat", "gi"]):
        site = g.site.iloc[0]
        base = df[(df.rat == rat) & (df.gi == (S2_BASE[(rat, site)] if s2_order else gi - 1))]
        if drop_base_trains:
            base = base[base.n_trains == 0]
        b = base[col].dropna()
        g = g[g[col].notna()]
        if stim_only:
            g = g[g.stimulated]
        for _, r in g.iterrows():
            out.append({"rat": rat, "gi": gi, "site": r.site, "target": r.target, "clip": r.clip, "dur": r[col],
                        "relative": r[col] / b.mean() - 1, "difference": r[col] - b.mean(),
                        "zscore": (r[col] - b.mean()) / b.std(ddof=1), "base_mean": b.mean(), "base_n": len(b),
                        "base_durs": b.values})
    return pd.DataFrame(out)


VARIANTS = {"main": dict(col="dur"), "stimulated only": dict(col="dur", stim_only=True),
            "baseline clips with trains left out": dict(col="dur", drop_base_trains=True),
            "thresholds (2, 1.5)": dict(col="dur_2_1.5"), "thresholds (4, 2.5)": dict(col="dur_4_2.5"),
            "Figure S2 group order": dict(col="dur", s2_order=True), "device end": dict(col="dur_dev")}
ND = {k: normalized(sz, **v) for k, v in VARIANTS.items()}
nd = ND["main"]
grp = nd.groupby(["rat", "site"]).agg(target=("target", "first"), n=("dur", "size"), dur=("dur", "mean"),
                                      base_mean=("base_mean", "first"), base_n=("base_n", "first"),
                                      relative=("relative", "mean"), sem=("relative", "sem"),
                                      difference=("difference", "mean"), zscore=("zscore", "mean")).reset_index()
print("Per stimulation group (main thresholds, all seizures): mean duration (s), baseline mean, and mean normalized duration")
print(grp.round(3).to_string(index=False))

### fig3a-target-reduces-each-rat

> In each of the 7 rats, stimulation of the target site reduced normalized seizure duration below
> the preceding baseline.

Criterion: reproduced when the mean normalized duration is below zero in all 8 target groups;
partially reproduced in 6 or 7.

In [ ]:
rows = []
for name, d in ND.items():
    g = d[d.target].groupby(["rat", "site"]).relative.mean()
    rows.append({"variant": name, "target_groups": len(g), "below_zero": int((g < 0).sum()),
                 **{f"{r}-{s}": round(v, 3) for (r, s), v in g.items()}})
print("Mean relative normalized duration of each target group (the sign is the same under all three")
print("normalization readings, as each subtracts the baseline mean):")
print(pd.DataFrame(rows).to_string(index=False))
print("Figure 3A, read from the bars: R1 DG -0.10, R2 DG -0.37, R2 CA1 -0.09, R3 CA3 -0.23, R4 CA3 -0.14, R5 ANT -0.33, R6 CA3 -0.21, R7 ANT -0.13")
for rat in RATS:
    print(rat, "seizures per target group:", nd[nd.target & (nd.rat == rat)].groupby("site").size().to_dict())

**Result: not reproduced.** Three of the 8 target groups are below their baseline (R3 CA3 -0.043,
R6 CA3 -0.257, R7 ANT -0.256); R1 DG (+0.011), R2 DG (+0.178), R2 CA1 (+0.092), R4 CA3 (+0.058)
and R5 ANT (+0.116) are above. The count is 3 for stimulated seizures only, without the baseline
clips with trains and with the Figure S2 group order (R5 ANT +0.255), 2 with thresholds (2, 1.5) and
with the device end, and 4 with thresholds (4, 2.5), short of the 6 the partial outcome needs.

### fig3a-nontarget-can-lengthen

> Stimulation of some non-target sites increased seizure duration above the preceding baseline.

Criterion: reproduced when at least 5 of the 20 non-target groups have a positive mean normalized
duration; not reproduced when fewer than 2 do.

In [ ]:
rows = []
for name, d in ND.items():
    g = d[~d.target].groupby(["rat", "site"]).relative.mean()
    rows.append({"variant": name, "non_target_groups": len(g), "positive": int((g > 0).sum()),
                 "which": ", ".join(f"{r} {s}" for (r, s), v in g.items() if v > 0)})
print(pd.DataFrame(rows).to_string(index=False))
print("Figure 3A, read from the bars, positive: R1 CA3 and ANT, R2 CA3, R3 DG and ANT, R4 DG and ANT, R5 CA1, R6 DG, R7 DG (10 of 20)")

**Result: reproduced.** Eleven of the 20 non-target groups have a positive mean normalized duration
(9 to 13 under the variants). Six of them are among the ten Figure 3A shows as positive (R2 CA3,
R3 DG and ANT, R4 DG and ANT, R5 CA1 are; R1 CA3 and ANT, R6 DG and R7 DG are not), and R4 CA1, R5
CA3, R6 ANT, R6 CA1 and R7 CA1 are positive here and not in the figure. The criterion's bar (5 of
20) is low: 5 of the 8 target groups are positive too (fig3a-target-reduces-each-rat), so this
count does not show that non-target stimulation lengthens seizures more than target stimulation.

### fig3a-per-rat-site-differences

> Within rats R2, R3, R4, R5 and R7 the four stimulation sites differ in normalized seizure duration
> (Kruskal-Wallis), while in R1 and R6 no pair of sites differs.

Criterion: Kruskal-Wallis per rat, then Dunn's test, (a) Bonferroni-adjusted or (b) unadjusted.
Reproduced when at least 8 of the 11 significant pairs of Figure 3A are significant and R1 and R6
have none; partially reproduced when at least 5 are.

In [ ]:
PAPER_PAIRS = {"R2": [("DG", "CA3"), ("CA3", "ANT")], "R3": [("DG", "CA1"), ("DG", "CA3"), ("CA3", "ANT")],
               "R4": [("DG", "CA1"), ("DG", "CA3")], "R5": [("CA1", "CA3"), ("CA1", "ANT")],
               "R7": [("DG", "CA3"), ("DG", "ANT")]}
rows, detail = [], []
for name in ["main", "stimulated only", "Figure S2 group order", "device end"]:
    d = ND[name]
    for norm in NORMS:
        for adjust in [True, False]:
            hit, r1r6 = 0, 0
            for rat in RATS:
                dr = d[d.rat == rat]
                kw = stats.kruskal(*[g[norm].values for _, g in dr.groupby("site")]).pvalue
                pw = dunn(dr[norm].values, dr.site.values, adjust)
                sig = {tuple(sorted(k)) for k, p in pw.items() if p < 0.05}
                if rat in PAPER_PAIRS:
                    hit += sum(tuple(sorted(p)) in sig for p in PAPER_PAIRS[rat])
                else:
                    r1r6 += len(sig)
                if name == "main":
                    detail.append({"Dunn": "Bonferroni" if adjust else "unadjusted", "norm": norm, "rat": rat, "KW_p": kw,
                                   "significant_pairs": ", ".join("-".join(k) for k in sorted(sig)) or "none",
                                   "paper_pairs": ", ".join("-".join(p) for p in PAPER_PAIRS.get(rat, [])) or "none"})
            rows.append({"seizures": name, "norm": norm, "Dunn": "Bonferroni" if adjust else "unadjusted",
                         "paper_pairs_found_of_11": hit, "pairs_in_R1_R6": r1r6})
print(pd.DataFrame(rows).to_string(index=False))
print("\nPer rat (all seizures), the significant pairs under each Dunn reading:")
print(pd.DataFrame(detail).round(4).to_string(index=False))

**Result: partially reproduced.** At most 5 of the 11 pairs of Figure 3A are significant in the
main group order (relative change or difference, unadjusted Dunn): R2 CA3-ANT, R3 DG-CA1 and
DG-CA3, R4 DG-CA1 and DG-CA3. Under these readings R1 and R6 have 4 to 6 significant pairs, where
the paper has none. With Bonferroni adjustment 1 to 3 of 11 are found; with the Figure S2 group
order, 6 unadjusted. The Kruskal-Wallis test is significant in R1, R2, R3 and R4 (relative change)
and not in R5 (p = 0.68), R6 (0.12) or R7 (0.62), so none of R5's and R7's four pairs is found. The
criterion's partial outcome needs only 5 of the 11 pairs, which the best reading reaches; the R1
and R6 part of the claim does not hold under any reading but the z-score with adjustment, which
finds 2 of the 11. Whether the partial outcome should require R1 and R6 to have no pairs is asked
of a reader (questions/yang-2025.yaml).

### fig3a-target-vs-nontarget-total

> Pooled over rats, target-site stimulation shortened normalized seizure duration (-0.26 +- 0.03)
> significantly more than non-target stimulation (0.038 +- 0.03).

Criterion: both means within tolerance and Mann-Whitney p < 0.001 with target lower; partially
reproduced when target is lower with p < 0.05.

In [ ]:
rows = []
for name, d in ND.items():
    for norm in NORMS:
        t, n = d[d.target][norm], d[~d.target][norm]
        rows.append({"variant": name, "norm": norm, "unit": "seizures", "n_target": len(t), "n_non_target": len(n),
                     "target_mean": t.mean(), "target_sem": t.sem(), "non_target_mean": n.mean(), "non_target_sem": n.sem(),
                     "p_MW": stats.mannwhitneyu(t, n).pvalue})
        gm = d.groupby(["rat", "site"]).agg(v=(norm, "mean"), target=("target", "first"))
        t, n = gm[gm.target].v, gm[~gm.target].v
        rows.append({"variant": name, "norm": norm, "unit": "groups", "n_target": len(t), "n_non_target": len(n),
                     "target_mean": t.mean(), "target_sem": t.sem(), "non_target_mean": n.mean(), "non_target_sem": n.sem(),
                     "p_MW": stats.mannwhitneyu(t, n).pvalue})
tot = pd.DataFrame(rows)
# the tolerances are in the units of the relative change (Figure 3A); the other readings are not judged on them
rel_ = tot.norm == "relative"
tot["target_ok"] = np.where(rel_, (tot.target_mean - -0.26).abs() <= 0.03, None)
tot["non_target_ok"] = np.where(rel_, (tot.non_target_mean - 0.038).abs() <= 0.03, None)
print("Paper: target -0.26 +- 0.03, non-target 0.038 +- 0.03, Mann-Whitney p < 0.001")
print(tot.round(4).to_string(index=False))
lower = tot[tot.target_mean < tot.non_target_mean]
print("Smallest p over all rows:", round(tot.p_MW.min(), 4), "; smallest with target lower:", round(lower.p_MW.min(), 4),
      "; rows with target lower at p < 0.05:", int((lower.p_MW < 0.05).sum()), "of", len(tot))

**Result: not reproduced.** Pooled over seizures, the relative change is -0.037 +- 0.033 for target
and +0.003 +- 0.038 for non-target stimulation (Mann-Whitney p = 0.93); over groups, -0.013 against
+0.006 (p = 0.94). No reading or variant gives target lower with p < 0.05 (smallest p with target
lower 0.25; smallest overall 0.096, difference from baseline at thresholds (2, 1.5), with target the
longer), and the target mean (relative change) is never near -0.26. The tolerances are judged only
for the relative change, the unit of Figure 3A.

### fig3a-some-nontarget-shorten

> Five non-target stimulations (R2 ANT, R3 CA1, R4 CA1, R5 CA3, R7 CA3) also significantly
> shortened seizure duration, but less than the target site of the same rat.

Criterion: readings of "significantly shortened", (a) Mann-Whitney against the preceding baseline,
(b) Wilcoxon of normalized durations against 0, (c) one-sample t-test; reproduced when at least 4 of
the 5 are significantly shorter and in at least 4 of the 5 rats the named group's mean is above that
of the target group (for R2, DG); partially reproduced when 3 of each hold.

In [ ]:
NAMED = [("R2", "ANT"), ("R3", "CA1"), ("R4", "CA1"), ("R5", "CA3"), ("R7", "CA3")]
rows = []
for name in ["main", "stimulated only", "Figure S2 group order", "device end"]:
    d = ND[name]
    for (rat, site), g in d[~d.target].groupby(["rat", "site"]):
        b = g.base_durs.iloc[0]
        tg = d[d.target & (d.rat == rat)]
        tmean = (tg[tg.site == "DG"] if rat == "R2" else tg).relative.mean()
        rows.append({"seizures": name, "group": f"{rat} {site}", "named": (rat, site) in NAMED, "n": len(g),
                     "relative_mean": g.relative.mean(), "target_mean_same_rat": tmean,
                     "less_effective_than_target": g.relative.mean() > tmean,
                     "p_MW_vs_baseline": stats.mannwhitneyu(g.dur, b, alternative="less").pvalue,
                     "p_wilcoxon_vs_0": stats.wilcoxon(g.relative, alternative="less").pvalue,
                     "p_t_vs_0": stats.ttest_1samp(g.relative, 0, alternative="less").pvalue,
                     "p_t_vs_0_two_sided": stats.ttest_1samp(g.relative, 0).pvalue,
                     "p_MW_two_sided": stats.mannwhitneyu(g.dur, b).pvalue,
                     "p_wilcoxon_two_sided": stats.wilcoxon(g.relative).pvalue})
sn = pd.DataFrame(rows)
for t in ["p_MW_vs_baseline", "p_wilcoxon_vs_0", "p_t_vs_0"]:
    sn[t.replace("p_", "sig_")] = sn[t] < 0.05
# two-sided: significant and shorter
for t in ["p_t_vs_0_two_sided", "p_MW_two_sided", "p_wilcoxon_two_sided"]:
    sn[t.replace("p_", "sig_")] = (sn[t] < 0.05) & (sn.relative_mean < 0)
print("One-sided tests (shorter than baseline). For R2 the target mean is that of the DG target, the more effective.")
print(sn.round(4).to_string(index=False))
for name in ["main", "stimulated only", "Figure S2 group order", "device end"]:
    s_ = sn[(sn.seizures == name) & sn.named]
    print(f"\n{name}: named groups significantly shorter (of 5):",
          {t: int(s_[t].sum()) for t in ["sig_MW_vs_baseline", "sig_wilcoxon_vs_0", "sig_t_vs_0", "sig_t_vs_0_two_sided",
                                         "sig_MW_two_sided", "sig_wilcoxon_two_sided"]},
          "; less effective than the rat's target:", int(s_.less_effective_than_target.sum()))
    o_ = sn[(sn.seizures == name) & ~sn.named]
    print("  other non-target groups significant by any test:",
          o_[o_.sig_MW_vs_baseline | o_.sig_wilcoxon_vs_0 | o_.sig_t_vs_0].group.tolist())

**Result: partially reproduced.** Under the one-sided t-test reading 3 of the 5 named groups are
significantly shorter than baseline (R2 ANT, R3 CA1, R7 CA3; 2 under the other two tests), and the
two-sided t-test gives the same 3. Three of the 5 are less effective than the rat's target (R4 CA1,
R5 CA3, R7 CA3), which meets the partial outcome. This rests on little: only R7 CA3 does both, R4
CA1 and R5 CA3 count as less effective only because the target group itself lengthened seizures
(+0.058 and +0.116), and R2 ANT (-0.271 against the DG target's +0.178) and R3 CA1 (-0.684 against
-0.043) shorten seizures more than the target. With the Figure S2 group order or the device end, 3
are shorter and 2 less effective. R1 CA1, R6 DG and R7 DG, which the paper does not name, are also
significantly shorter.

### fig3b-same-site-target-vs-nontarget

> For the same site, stimulation as a target shortened normalized seizure duration more than
> stimulation of that site as a non-target, significantly for DG, CA3 and ANT but not CA1.

Criterion: per site, Mann-Whitney of target against non-target seizures pooled over rats (unpaired
t-test as a variant); reproduced when DG, CA3 and ANT each show target lower with p < 0.05;
partially reproduced for two of the three.

In [ ]:
rows = []
for name in ["main", "stimulated only", "Figure S2 group order", "device end"]:
    d = ND[name]
    for norm in NORMS:
        for site in y.CHANNELS:
            t = d[d.target & (d.site == site)][norm]
            n = d[~d.target & (d.site == site)][norm]
            rows.append({"seizures": name, "norm": norm, "site": site, "n_target": len(t), "n_non_target": len(n),
                         "target_mean": t.mean(), "non_target_mean": n.mean(),
                         "p_MW": stats.mannwhitneyu(t, n).pvalue, "p_t": stats.ttest_ind(t, n).pvalue})
f3b = pd.DataFrame(rows)
f3b["holds"] = (f3b.target_mean < f3b.non_target_mean) & (f3b.p_MW < 0.05)
print(f3b.round(4).to_string(index=False))
print("\nPer reading, of DG, CA3 and ANT, sites with target lower at MW p < 0.05:")
print(f3b[f3b.site != "CA1"].groupby(["seizures", "norm"]).holds.sum().to_string())

**Result: not reproduced.** No site shows target lower with p < 0.05 under any normalization,
seizure set, group order or end reading. With relative change, target is lower at CA3 (-0.096
against +0.026, p = 0.51), ANT (-0.094 against +0.093, p = 0.19) and DG (+0.069 against +0.083,
p = 0.63), none significantly. At CA1, the part of the claim that expects no
difference, p = 0.079, so that part holds.

## PSD ratio

Welch PSD band ratios (Methods D) in the pre and post windows: for stimulation groups, onset to the
first train, and the 5 s after the first train ends (assumption); for baseline groups, 0 to 5 s and
10 to 15 s after onset. Three window readings (`yang2025.psd_windows`): "guarded", the main one,
keeps 0.25 s clear of each train and ends the post window where the next train starts; "paper"
takes Methods D as worded; "guarded_pre5" limits the pre window to the 5 s before the first train. Readings of the claims file: change measure (relative, log2 ratio,
difference), 1 s or 0.5 s Hann segments with 50% overlap, total 0.5-120 Hz or 0-500 Hz, all
channels or the stimulated channel (for a baseline seizure, the channel of the site stimulated in the
next group, so that each condition gives one channel per seizure), and Dunn's post hoc test
adjusted or not. The unit is the seizure-channel pair.

In [ ]:
BAND_NAMES = list(y.BANDS)
WREAD = list(y.WINDOW_READINGS)  # "guarded" is the main window reading (yang2025.psd_windows)
prow = []
for c in clips:
    if not c.get("psd_pre"):
        continue
    cond = "baseline" if c["kind"] == "baseline" else ("target" if (c["rat"], c["site"]) in S2_TARGETS else "non-target")
    # the stimulated channel; for a baseline seizure, the site stimulated in the next group (dataset order),
    # so that every condition gives one channel per seizure under that reading
    site = c["site"] if c["kind"] == "stim" else stim_groups[(c["rat"], c["index"] + 1)]
    for wr, pres in c["psd_pre"].items():
        for key in pres:
            pre, post = pres[key], c["psd_post"][wr][key]
            for ch in range(4):
                for b, band in enumerate(BAND_NAMES):
                    prow.append({"window": wr, "rat": c["rat"], "group": c["group"], "clip": c["clip"], "condition": cond,
                                 "stim_channel": site == y.CHANNELS[ch], "channel": y.CHANNELS[ch],
                                 "nperseg": key[0], "total": key[1], "band": band, "pre": pre[ch, b], "post": post[ch, b]})
psd = pd.DataFrame(prow)
psd["rel"] = (psd.post - psd.pre) / psd.pre
psd["log2"] = np.log2(psd.post / psd.pre)
psd["diff"] = psd.post - psd.pre
n_all = sz.groupby("condition").size()
for wr in WREAD:
    n_with = psd[psd.window == wr].drop_duplicates(["clip", "group", "rat"]).groupby("condition").size()
    print(f"Window reading {wr}: seizures with both PSD windows (each at least 0.5 s):", n_with.to_dict(), "of", n_all.to_dict())
for wr in WREAD:
    pw = [c["psd_windows"][wr] for c in clips if c["kind"] == "stim" and c.get("psd_windows", {}).get(wr)]
    pre_len = np.array([w[0][1] - w[0][0] for w in pw])
    post_len = np.array([w[1][1] - w[1][0] for w in pw])
    print(f"{wr}: stimulation-group pre window, s:", pd.Series(pre_len).describe().round(2).to_dict(),
          "; shorter than 0.5 s (dropped):", int((pre_len < 0.5).sum()), "; shorter than 1 s (Welch segment shortened):",
          int((pre_len < 1).sum()), "; post windows shorter than 0.5 s (dropped):", int((post_len < 0.5).sum()))
print("PSD ratio ranges (pre, guarded):", psd[psd.window == "guarded"].groupby("band").pre.agg(["min", "median", "max"]).round(4).to_dict("index"))


### fig4b-psd-target-bands

> After target stimulation the change in PSD ratio was larger in low bands (alpha, and theta or
> beta) and more negative in gamma than after sham (baseline) or non-target stimulation.

Criterion: reproduced when, under one reading of the method and one of the bands, target differs
from both sham and non-target in the named direction with p < 0.05 in all three bands; partially
reproduced when alpha and gamma hold.

In [ ]:
def band_tests(d, measure, adjust):
    out = {}
    for band in BAND_NAMES:
        x = d[d.band == band]
        groups = {k: g[measure].replace([np.inf, -np.inf], np.nan).dropna().values for k, g in x.groupby("condition")}
        kw = stats.kruskal(*groups.values()).pvalue
        vals = np.concatenate(list(groups.values()))
        labs = np.concatenate([[k] * len(v) for k, v in groups.items()])
        pwd = dunn(vals, labs, adjust)
        g = lambda a, b: pwd.get((a, b), pwd.get((b, a)))
        out[band] = {"KW_p": kw, "target_vs_baseline_p": g("target", "baseline"), "target_vs_nontarget_p": g("target", "non-target"),
                     **{f"mean_{k}": np.mean(v) for k, v in groups.items()}, **{f"median_{k}": np.median(v) for k, v in groups.items()}}
    return out


READINGS_BANDS = {"(i) theta, alpha up, gamma down": {"theta": 1, "alpha": 1, "gamma": -1},
                  "(ii) alpha, beta up, gamma down": {"alpha": 1, "beta": 1, "gamma": -1}}
rows, store = [], {}
for wr in WREAD:
    for nps in y.PSD_NPERSEG:
        for total in y.PSD_TOTALS:
            for chans in ["all", "stimulated"]:
                d = psd[(psd.window == wr) & (psd.nperseg == nps) & (psd.total == total)]
                if chans == "stimulated":
                    d = d[d.stim_channel]
                for measure in ["rel", "log2", "diff"]:
                    for adjust in [True, False]:
                        bt = band_tests(d, measure, adjust)
                        store[(wr, nps, total, chans, measure, adjust)] = bt
                        for rname, want in READINGS_BANDS.items():
                            ok, which = 0, []
                            for band, sign in want.items():
                                r = bt[band]
                                direction = np.sign(r["median_target"] - r["median_baseline"]) == sign and \
                                    np.sign(r["median_target"] - r["median_non-target"]) == sign
                                hold = bool(direction and r["target_vs_baseline_p"] < 0.05 and r["target_vs_nontarget_p"] < 0.05)
                                ok += hold
                                which += [band] if hold else []
                            alpha_gamma = all(
                                np.sign(bt[b]["median_target"] - bt[b]["median_baseline"]) == s and
                                np.sign(bt[b]["median_target"] - bt[b]["median_non-target"]) == s and
                                bt[b]["target_vs_baseline_p"] < 0.05 and bt[b]["target_vs_nontarget_p"] < 0.05
                                for b, s in [("alpha", 1), ("gamma", -1)])
                            rows.append({"window": wr, "nperseg": nps, "total": total, "channels": chans, "measure": measure,
                                         "Dunn": "Bonf" if adjust else "unadj", "bands": rname, "bands_holding_of_3": ok, "which": ",".join(which),
                                         "alpha_and_gamma": alpha_gamma})
f4 = pd.DataFrame(rows)
print("Per reading: bands (of the 3 named) where target differs from both sham and non-target in the named direction")
print("(medians, Dunn p < 0.05), and whether alpha and gamma both hold:")
print(f4.groupby(["window", "bands"]).bands_holding_of_3.value_counts().to_string())
for wr in WREAD:
    f_ = f4[f4.window == wr]
    print(f"{wr}: readings where alpha and gamma both hold:", int(f_.alpha_and_gamma.sum()), "of", len(f_),
          "; readings where each band holds:", {b: int(f_.which.str.contains(b).sum()) for b in ["theta", "alpha", "beta", "gamma"]})
print(f4.sort_values("bands_holding_of_3", ascending=False).head(12).to_string(index=False))
print("\nMain reading shown in full (guarded windows, 1 s segments, total 0.5-120 Hz, all channels, relative change, Dunn with Bonferroni):")
print(pd.DataFrame(store[("guarded", 1000, "0.5-120", "all", "rel", True)]).T.round(4).to_string())
print("\nThe same with log2(post/pre):")
print(pd.DataFrame(store[("guarded", 1000, "0.5-120", "all", "log2", True)]).T.round(4).to_string())
print("\nThe same (relative change) with the pre window limited to the 5 s before the first train:")
print(pd.DataFrame(store[("guarded_pre5", 1000, "0.5-120", "all", "rel", True)]).T.round(4).to_string())
print("\nBy rat, median relative change of the alpha and gamma ratio (guarded, 1 s, 0.5-120 Hz, all channels):")
d = psd[(psd.window == "guarded") & (psd.nperseg == 1000) & (psd.total == "0.5-120") & psd.band.isin(["alpha", "gamma"])]
print(d.groupby(["band", "rat", "condition"]).rel.median().unstack().round(3).to_string())

**Result: not reproduced.** Of the 288 readings (three window readings, change measure, Welch
segment, total range, channels, post hoc adjustment, and the two band readings), none has alpha and
gamma both holding, and at most 2 of the 3 named bands hold (beta and gamma, under the difference
measure with total 0-500 Hz, the stimulated channel and unadjusted Dunn). Alpha and theta hold
under no reading; with the pre window limited to 5 s no band holds under any. In the main reading
(guarded windows, 1 s segments, 0.5-120 Hz, all channels, relative change), the alpha ratio rises
after target stimulation (median +0.51) more than after non-target (+0.16, p = 0.005) but not more
than after sham (+0.58, p = 0.81), and the gamma ratio falls after target (-0.21) about as much as
after sham (-0.19, p = 1.0), more than after non-target (-0.10, p = 0.010). Theta falls after
target stimulation (-0.16). The guarded windows hold no train, so the aliased 130 Hz harmonics
(multiples of 10 Hz at 1 kHz) do not enter them; the windows as worded give the same picture.

### fig4b-psd-other-bands-unchanged

> In the other bands (delta and high gamma, and whichever of theta and beta is not named as changed)
> the change in PSD ratio did not differ between conditions.

Criterion: a claim of absence, Kruskal-Wallis across sham, target and non-target at p < 0.05; delta
and high gamma judged in every reading, beta under reading (i) and theta under (ii). As
fig4b-psd-target-bands was decided by neither band reading, both are shown.

In [ ]:
rows = []
for key, bt in store.items():
    if not key[5]:
        continue
    for band in ["delta", "high_gamma", "beta", "theta"]:
        rows.append({"window": key[0], "nperseg": key[1], "total": key[2], "channels": key[3], "measure": key[4], "band": band,
                     "KW_p": bt[band]["KW_p"], "median_baseline": bt[band]["median_baseline"],
                     "median_target": bt[band]["median_target"], "median_non_target": bt[band]["median_non-target"]})
f4o = pd.DataFrame(rows)
print("Kruskal-Wallis across sham (baseline), target and non-target, per band and reading:")
print(f4o.pivot_table(index=["window", "nperseg", "total", "channels", "measure"], columns="band", values="KW_p").round(4).to_string())
print("\nReadings with KW p >= 0.05, per band and window reading (of", len(f4o) // 4 // len(WREAD), "each):")
print(f4o.assign(ns=f4o.KW_p >= 0.05).groupby(["window", "band"]).ns.sum().unstack().to_string())
pv = f4o.pivot_table(index=["window", "nperseg", "total", "channels", "measure"], columns="band", values="KW_p")
for rname, judged in [("(i)", ["delta", "high_gamma", "beta"]), ("(ii)", ["delta", "high_gamma", "theta"])]:
    ok = (pv[judged] >= 0.05).all(axis=1)
    print(f"Band reading {rname}: readings with no judged band significant:", int(ok.sum()), "of", len(pv), pv[ok].index.tolist())
print("\nMain reading (guarded, 1 s, 0.5-120 Hz, all channels, relative change):")
print(f4o[(f4o.window == "guarded") & (f4o.nperseg == 1000) & (f4o.total == "0.5-120") & (f4o.channels == "all") & (f4o.measure == "rel")].round(4).to_string(index=False))

**Result: not reproduced.** In the main reading the Kruskal-Wallis test is significant for delta
(p < 0.0001; medians sham -0.17, target -0.55, non-target -0.35), high gamma (p = 0.0006), beta
(p = 0.0001) and theta (p < 0.0001), so the claim fails under both band readings. No reading of the
72 leaves all judged bands without a difference, under either band reading. High gamma differs
under every reading but 2 (both with the pre window limited to 5 s), and delta under all but 11 of
72.

### fig5b-iis-post-seizure

> The IIS rate 15 s to 45 s after seizures was lower after target stimulation than after non-target
> stimulation.

Criterion: IIS as Methods F on all channels, threshold from the 180 s before onset; readings (a)
Mann-Whitney of per seizure-channel mean rates over 15 to 45 s, (b) paired t-test across time bins
of 15 to 45 s (1 s or 5 s bins), target curve against non-target curve. Reproduced when, under one
reading, target is lower with p < 0.001; partially reproduced with p < 0.05.

In [ ]:
# IIS rate after the seizure end, per seizure and channel, in the stimulation groups. Spikes within
# IIS_GUARD of a stimulation train are dropped and that time is left out of the rate, as the train
# artefact makes false spikes (verification round 1); the round-1 count without the mask is a variant.
IIS_GUARD = 0.5  # s [standard]


def masked_rate(t, lo, hi, spans):
    """Spikes per s at times t (s, relative to the reference) in [lo, hi), leaving out spans."""
    grid = np.arange(lo, hi, 0.01) + 0.005
    ok = np.ones(len(grid), bool)
    keep = np.ones(len(t), bool)
    for a, b in spans:
        ok &= ~((grid >= a) & (grid < b))
        keep &= ~((t >= a) & (t < b))
    cover = ok.mean() * (hi - lo)
    return (((t >= lo) & (t < hi) & keep).sum() / cover) if cover > 0 else np.nan, cover


irow = []
for c in clips:
    if c["kind"] != "stim" or "iis_post" not in c:
        continue
    s = c["seizures"][MAIN]
    end = s["end"]
    last = c["trains"][-1][1] if c["trains"] else end
    shift = max(0.0, last - end)  # device-end variant: times relative to max(end, last train end)
    span_post = c["iis_span"][1]
    spans = [(a - IIS_GUARD - end, b + IIS_GUARD - end) for a, b in c["trains"]]
    tgt = (c["rat"], c["site"]) in S2_TARGETS
    for ch in range(4):
        post = c["iis_post"][ch]
        pre = c["iis_pre"][ch]
        full = span_post >= 45
        r_m, cov = masked_rate(post, 15, 45, spans) if full else (np.nan, np.nan)
        r_d = masked_rate(post - shift, 15, 45, [(a - shift, b - shift) for a, b in spans])[0] if span_post - shift >= 45 else np.nan
        irow.append({"rat": c["rat"], "group": c["group"], "clip": c["clip"], "channel": y.CHANNELS[ch], "target": tgt,
                     "post_span": span_post, "rate_15_45": r_m, "covered_s": cov,
                     "rate_15_45_unmasked": ((post >= 15) & (post < 45)).sum() / 30 if full else np.nan,
                     "rate_15_45_device_end": r_d,
                     "rate_pre": len(pre) / -c["iis_span"][0], "post": post, "spans": spans})
iis = pd.DataFrame(irow)
n_stim = int((sz.kind == "stim").sum())
print("Stimulation-group seizures with IIS detected (at least 30 s before onset in the clip):",
      iis.drop_duplicates(["rat", "group", "clip"]).shape[0], "of", n_stim,
      "; with the full 15-45 s after the end in the clip:", iis[iis.rate_15_45.notna()].drop_duplicates(["rat", "group", "clip"]).shape[0],
      "; with 15-45 s after the device end:", iis[iis.rate_15_45_device_end.notna()].drop_duplicates(["rat", "group", "clip"]).shape[0])
print("Time after the seizure end available in the clip (s):", iis.drop_duplicates(["rat", "group", "clip"]).post_span.describe().round(1).to_dict())
d0 = iis[iis.rate_15_45.notna()].drop_duplicates(["rat", "group", "clip"])
print("Seizures whose 15-45 s window loses time to a train:", d0.groupby("target").apply(lambda g: f"{int((g.covered_s < 30).sum())} of {len(g)}").to_dict(),
      "; time left out, s:", round(float((30 - d0.covered_s).sum()), 1))
print("IIS rate (per s) in the 180 s before onset, per seizure and channel:", iis.rate_pre.describe().round(3).to_dict())


def reading_a(col, dd):
    d = dd[dd[col].notna()]
    t, n = d[d.target][col], d[~d.target][col]
    ds = d.groupby(["rat", "group", "clip", "target"])[col].mean().reset_index()
    t2, n2 = ds[ds.target][col], ds[~ds.target][col]
    return {"target": t.mean(), "target_sem": t.sem(), "n_target": len(t), "non_target": n.mean(), "non_target_sem": n.sem(),
            "n_non_target": len(n), "p_MW": stats.mannwhitneyu(t, n).pvalue,
            "per_seizure_target": t2.mean(), "per_seizure_non_target": n2.mean(), "per_seizure_p_MW": stats.mannwhitneyu(t2, n2).pvalue}


def reading_b(dd, bw, ref_shift=False):
    edges = np.arange(0, 180 + bw, bw)
    curves = {}
    for tg in [True, False]:
        counts = np.zeros(len(edges) - 1); cover = np.zeros(len(edges) - 1)
        for _, r in dd[dd.target == tg].iterrows():
            post, spans = r.post, r.spans
            keep = np.ones(len(post), bool)
            for a, b in spans:
                keep &= ~((post >= a) & (post < b))
            h, _ = np.histogram(post[keep], edges)
            grid = np.arange(0, 180, 0.01) + 0.005
            okg = np.ones(len(grid), bool)
            for a, b in spans:
                okg &= ~((grid >= a) & (grid < b))
            frac = np.histogram(grid[okg], edges)[0] / np.histogram(grid, edges)[0]
            inclip = edges[1:] <= r.post_span
            counts += np.where(inclip, h, 0); cover += np.where(inclip, frac * bw, 0)
        curves[tg] = counts / np.maximum(cover, 1e-9)
    sel = (edges[:-1] >= 15) & (edges[1:] <= 45)
    tt = stats.ttest_rel(curves[True][sel], curves[False][sel])
    return curves[True][sel].mean(), curves[False][sel].mean(), tt.statistic, tt.pvalue, int(sel.sum())


rows = []
for label, dd in [("all rats", iis), ("without R7", iis[iis.rat != "R7"])]:
    for col, name in [("rate_15_45", "trains masked (main)"), ("rate_15_45_unmasked", "no mask (round 1)"),
                      ("rate_15_45_device_end", "trains masked, device end")]:
        rows.append({"rats": label, "reading": "(a) " + name, **reading_a(col, dd)})
print("\nReading (a): per seizure-channel mean rate over 15-45 s after the end, Mann-Whitney; and per seizure (channels averaged):")
print(pd.DataFrame(rows).round(4).to_string(index=False))
print("\nReading (b): paired t-test across bins of 15-45 s, target curve against non-target curve (trains masked):")
for label, dd in [("all rats", iis), ("without R7", iis[iis.rat != "R7"])]:
    for bw in [1, 5]:
        mt, mn, tstat, p, nb_ = reading_b(dd, bw)
        print(f"  {label}, {bw} s bins: target {mt:.3f}, non-target {mn:.3f} per s; t = {tstat:.2f} over {nb_} bins, p = {p:.3g}")
print("\nBy rat, reading (a) means (per s), trains masked; and the per-rat Mann-Whitney p:")
d = iis[iis.rate_15_45.notna()]
br = d.groupby(["rat", "target"]).rate_15_45.mean().unstack()
br["p_MW"] = [stats.mannwhitneyu(d[(d.rat == r) & d.target].rate_15_45, d[(d.rat == r) & ~d.target].rate_15_45).pvalue for r in br.index]
br["target_lower"] = br[True] < br[False]
print(br.round(4).to_string())
print("Rats where target is lower:", int(br.target_lower.sum()), "of", len(br))
print("\nBaseline-threshold variant is not run: the interictal segments of reading (b) are not in the dataset.")


**Result: reproduced under reading (b).** With spikes within 0.5 s of a train dropped and that time
left out (7 of the 249 seizures lose time in the window, 53.6 s in all), the IIS rate over 15 to 45
s after the seizure end is 0.171 per s after target and 0.261 per s after non-target stimulation
across 1 s bins, paired t-test p = 1.25e-08 (5 s bins, p = 0.011). Under reading (a), per
seizure-channel, 0.183 against 0.277, Mann-Whitney p = 0.0085, and per seizure p = 0.038. Without
the mask the numbers barely change (0.179 against 0.286, p = 0.0051), so train artefact does not
make the difference. The result rests on R7. Its non-target rate (1.12 per s) is far above every
other rat's, and without R7 the time-bin test is not significant (0.139 against 0.152, p = 0.16;
5 s bins p = 0.47), while reading (a) still gives p = 0.0019 for a small difference. Target is lower
in 4 of the 7 rats (R1, R2, R6, R7) and higher in R3, R4 and R5. With the device end, reading (a)
gives 0.142 against 0.211 (p = 0.0010). Bins of one curve are not independent samples, so the
p-value of reading (b) overstates the evidence. The outcome rests on the onset and end rule and on
the 180 s pre-seizure baseline, as the interictal segments of the other baseline reading are not in
the dataset.

## HFOs during seizures

Ripples (80-250 Hz) and fast ripples (250-490 Hz) are counted by the short line length detector of
the HFO App (ref. 50) at its defaults, as the claims file says (`yang2025.hfo_events`, following
`HFOAutoDetectorSLL.m` of github.com/zelanolab/hfoapp at commit 1ccff19: 5 ms window, 97.5th
percentile threshold over the epoch, here the clip, events longer than 12 ms). In place of the
manual review, events within 0.25 s of a train are removed as artefact; a variant takes the
threshold from the samples outside the trains, as the whole-clip threshold includes the train
artefact. Counts are taken in the PSD windows under the three window readings. Per seizure, the
rates are summed over the channels used (all, or the stimulated one; for baseline seizures the site
stimulated next), normalized as post / pre - 1 (Figure 6 legend). Seizures with no event in the pre
window have no normalized value and are dropped (counts printed); a pooled variant, events over time
in each rat's post windows against its pre windows, drops none. Pairing (a) by rat, each condition's
mean per rat, paired t-test; (b) unpaired t-test across seizures. No manual review.

In [ ]:
hrow = []
for c in clips:
    if not c.get("hfo_counts"):
        continue
    cond = "baseline" if c["kind"] == "baseline" else ("target" if (c["rat"], c["site"]) in S2_TARGETS else "non-target")
    site = c["site"] if c["kind"] == "stim" else stim_groups[(c["rat"], c["index"] + 1)]
    for wr, counts in c["hfo_counts"].items():
        (a0, a1), (b0, b1) = c["psd_windows"][wr]
        if a1 - a0 < 0.5 or b1 - b0 < 0.5:
            continue
        for (band, th), cnt in counts.items():
            for ch in range(4):
                hrow.append({"window": wr, "rat": c["rat"], "group": c["group"], "clip": c["clip"], "condition": cond, "band": band,
                             "thresh": th, "channel": y.CHANNELS[ch], "stim_channel": site == y.CHANNELS[ch],
                             "pre_n": cnt[ch, 0], "post_n": cnt[ch, 1], "pre_s": a1 - a0, "post_s": b1 - b0})
hfo = pd.DataFrame(hrow)
hfo["pre_rate"], hfo["post_rate"] = hfo.pre_n / hfo.pre_s, hfo.post_n / hfo.post_s
print("Seizures with HFO counts, per window reading:",
      {wr: g.drop_duplicates(["rat", "group", "clip"]).groupby("condition").size().to_dict() for wr, g in hfo.groupby("window")})
print("HFO rates (events/s) per channel in the pre window (guarded), by band and threshold reading:")
print(hfo[hfo.window == "guarded"].groupby(["band", "thresh"]).pre_rate.describe().round(2).to_string())
print("Post window (guarded):")
print(hfo[hfo.window == "guarded"].groupby(["band", "thresh"]).post_rate.describe().round(2).to_string())


def hfo_norm(band, thresh, chans, window="guarded"):
    d = hfo[(hfo.band == band) & (hfo.thresh == thresh) & (hfo.window == window)]
    if chans == "stimulated":
        d = d[d.stim_channel]
    # per seizure: rates summed over the channels used, then post / pre - 1 (Figure 6 legend: ictal data
    # normalized to pre-stimulation data)
    s = d.groupby(["rat", "group", "clip", "condition"]).agg(pre_n=("pre_n", "sum"), post_n=("post_n", "sum"),
                                                             pre_s=("pre_s", "first"), post_s=("post_s", "first")).reset_index()
    s["pre_rate"], s["post_rate"] = s.pre_n / s.pre_s, s.post_n / s.post_s
    s["norm"] = np.where(s.pre_rate > 0, s.post_rate / s.pre_rate - 1, np.nan)
    return s


def hfo_tests(s):
    out = {"n_dropped_pre_zero": int(s.norm.isna().sum()), "n_used": int(s.norm.notna().sum())}
    # variant: pooled per rat and condition (events over time in the post windows against the pre windows), which
    # drops no seizure and weights short pre windows less
    pooled = s.groupby(["rat", "condition"]).apply(lambda g: (g.post_n.sum() / g.post_s.sum()) / (g.pre_n.sum() / g.pre_s.sum()) - 1
                                                   if g.pre_n.sum() > 0 else np.nan).unstack()
    s = s.dropna(subset=["norm"])
    per_rat = s.groupby(["rat", "condition"]).norm.mean().unstack()
    for k in ["baseline", "target", "non-target"]:
        out[f"mean_{k}"] = s[s.condition == k].norm.mean()
        out[f"sem_{k}"] = s[s.condition == k].norm.sem()
        out[f"pooled_mean_{k}"] = pooled[k].mean()
    for a, b in [("target", "non-target"), ("target", "baseline")]:
        pr = per_rat[[a, b]].dropna()
        out[f"paired_{a}_vs_{b}_p"] = stats.ttest_rel(pr[a], pr[b]).pvalue
        out[f"paired_{a}_vs_{b}_diff"] = (pr[a] - pr[b]).mean()
        out[f"unpaired_{a}_vs_{b}_p"] = stats.ttest_ind(s[s.condition == a].norm, s[s.condition == b].norm).pvalue
        pp = pooled[[a, b]].dropna()
        out[f"pooled_paired_{a}_vs_{b}_p"] = stats.ttest_rel(pp[a], pp[b]).pvalue
        out[f"pooled_paired_{a}_vs_{b}_diff"] = (pp[a] - pp[b]).mean()
    return out, per_rat


### fig6e-ictal-ripples-no-difference

> During seizures, ripple rates normalized to pre-stimulation did not differ between target and
> non-target stimulation.

Criterion: a claim of absence, the paired t-test at p < 0.05 under pairing (a).

In [ ]:
rows = []
for wr in WREAD:
    for th in y.HFO_THRESH:
        for chans in ["all", "stimulated"]:
            o, pr = hfo_tests(hfo_norm("ripple", th, chans, wr))
            rows.append({"window": wr, "thresh": th, "channels": chans, **o})
f6e = pd.DataFrame(rows)
print("Ictal ripple rate, post / pre - 1, per seizure. Paired t-test by rat (reading a) and unpaired across seizures (b):")
print(f6e.round(4).T.to_string())
print("Readings with paired (a) p < 0.05, target against non-target:", int((f6e["paired_target_vs_non-target_p"] < 0.05).sum()), "of", len(f6e),
      "; pooled variant:", int((f6e["pooled_paired_target_vs_non-target_p"] < 0.05).sum()))
o, pr = hfo_tests(hfo_norm("ripple", "clip", "all"))
print("\nPer rat means (main reading: guarded windows, threshold over the clip, all channels):")
print(pr.round(3).to_string())

**Result: reproduced.** Under the paired reading, in the main reading (guarded windows, threshold
over the clip, all channels) target and non-target do not differ (3.31 against 12.84, p = 0.17), nor
in any of the 6 readings with all channels (p = 0.17 to 0.96) or in any pooled variant (p = 0.07 to
0.81). With the stimulated channel and the threshold taken outside the trains, 2 of the 12
readings give p < 0.05 (0.037 and 0.039, target lower), so the absence holds under every reading
with all channels and fails under 2 with the stimulated channel; the outcome rests on that reading.
The normalized rates are large and positive in every condition (baseline 12.1), far from the -0.16
and -0.04 read from Figure 6E; 71 to 266 seizures with no ripple in the pre window are dropped. With
7 rat pairs and a detector without the manual review, this is weak evidence for the absence.

### fig6f-ictal-fast-ripples-reduced

> During seizures, target stimulation reduced the fast ripple rate, normalized to pre-stimulation,
> compared with baseline and with non-target stimulation.

Criterion: reproduced when, under one pairing reading, the target mean is below both others with
p < 0.05 for each comparison; partially reproduced when one comparison holds.

In [ ]:
rows = []
for wr in WREAD:
    for th in y.HFO_THRESH:
        for chans in ["all", "stimulated"]:
            o, pr = hfo_tests(hfo_norm("fast_ripple", th, chans, wr))
            rows.append({"window": wr, "thresh": th, "channels": chans, **o})
f6f = pd.DataFrame(rows)
print("Ictal fast ripple rate, post / pre - 1, per seizure:")
print(f6f.round(4).T.to_string())
both = (f6f.mean_target < f6f.mean_baseline) & (f6f.mean_target < f6f["mean_non-target"])
print("Readings with the target mean below both:", int(both.sum()), "of", len(f6f),
      "; with paired p < 0.05 against non-target:", int((f6f["paired_target_vs_non-target_p"] < 0.05).sum()),
      "; against baseline:", int((f6f["paired_target_vs_baseline_p"] < 0.05).sum()),
      "; either, with target lower:", int((((f6f["paired_target_vs_non-target_p"] < 0.05) & (f6f["paired_target_vs_non-target_diff"] < 0)) |
                                            ((f6f["paired_target_vs_baseline_p"] < 0.05) & (f6f["paired_target_vs_baseline_diff"] < 0))).sum()))
print("Pooled variant, either comparison with target lower at p < 0.05:", int((((f6f["pooled_paired_target_vs_non-target_p"] < 0.05) & (f6f["pooled_paired_target_vs_non-target_diff"] < 0)) |
                                            ((f6f["pooled_paired_target_vs_baseline_p"] < 0.05) & (f6f["pooled_paired_target_vs_baseline_diff"] < 0))).sum()))
o, pr = hfo_tests(hfo_norm("fast_ripple", "clip", "all"))
print("\nPer rat means (main reading: guarded windows, threshold over the clip, all channels):")
print(pr.round(3).to_string())

**Result: not reproduced.** In the main reading (guarded windows, threshold over the clip, all
channels) the normalized fast ripple rate is 6.25 after target, 12.58 for baseline and 9.26 after
non-target stimulation, but the paired tests give p = 0.90 against baseline and 0.50 against
non-target. The target mean is below both others in 5 of the 12 readings, and no reading, and no
pooled variant, has either comparison significant with target lower (all paired p >= 0.13).
Between 143 and 308 seizures with no fast ripple in the pre window are dropped, and the means are
far from the -0.25, 0.2 and 0.1 read from Figure 6F.

## Summary

| Claim | Outcome |
|---|---|
| tables1-network-stable-short-term | reproduced (only with self-pairs included), assumed |
| fig2-r1-target-shifts | partially reproduced, assumed |
| figs2-target-assignment | partially reproduced, assumed |
| fig3a-target-reduces-each-rat | not reproduced, assumed |
| fig3a-nontarget-can-lengthen | reproduced, assumed |
| fig3a-per-rat-site-differences | partially reproduced, assumed |
| fig3a-target-vs-nontarget-total | not reproduced, assumed |
| fig3a-some-nontarget-shorten | partially reproduced, assumed |
| fig3b-same-site-target-vs-nontarget | not reproduced, assumed |
| seizure-duration-proportion | blocked, data missing |
| fig3c-frequency-no-difference | blocked, data missing |
| fig4b-psd-target-bands | not reproduced, assumed |
| fig4b-psd-other-bands-unchanged | not reproduced, assumed |
| fig5b-iis-post-seizure | reproduced (time-bin reading; not without R7), assumed |
| fig5c-interictal-iis-no-difference | blocked, data missing |
| fig5d-iis-seizure-correlation | blocked, data missing |
| fig6cd-interictal-hfo-no-difference | blocked, data missing |
| fig6e-ictal-ripples-no-difference | reproduced, assumed |
| fig6f-ictal-fast-ripples-reduced | not reproduced, assumed |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

How the analysis differs from the paper's. The dandiset holds no seizure onset or end times, so
they are marked here by a line-length rule; every tested outcome rests on it. The clips are not cut
at a fixed onset (the first train falls at 292 to 328 s, 25th to 75th percentile), so the clip centre
cannot stand in for one. The stimulation times are found from the artefact. The PSD post window is
read as the 5 s after the first train ends, kept 0.25 s clear of the trains and cut at the next
train. HFOs are counted by the HFO App's short line length detector at its defaults, with events
near trains removed in place of the manual review. IIS near trains are dropped. GC is a numpy reimplementation of the MVGC time-domain estimator. The normalization
formula, the unit of analysis, the post hoc test and the pairing of the paired t-tests are not given
by the paper; every reading the claims file names is shown. The revision after verification round 1
added these checks and variants; it changed no outcome.

What was found about the dataset. It holds the ictal clips of all 7 rats and 56 groups, but no
seizure or stimulation times, no interictal recordings and no recording durations, which blocks 5
claims and leaves every other one resting on an assumed onset rule. Eleven baseline clips carry
stimulation trains, at least seven of them like those of the stimulation groups (R1 base2, R4 base0,
R5 base0, R6 base1). Sixteen clip names occur in two groups of the same rat with different
contents, and some groups' clip dates fall outside their place in the group order. The stimulation
order of R5 and R6 differs from Figure S2. Two groups hold fewer than 6 seizures. The `-small` and
`-large` clip names are unexplained, the subject age and weight are the same in all sessions, and
the ANT depth differs from the paper's.

In [ ]:
print(f"Notebook run time: {(time.time() - started) / 60:.1f} min")